# Third approach: frozen MMS-1B encoder + trained CTC head
Run these cells in order on a Colab GPU. This is a distinct training strategy from the existing MMS top-four-layer fine-tuning. It uses the **same saved 531/124/114 FLEURS split**. The held-out test is evaluated only after validation selects a checkpoint. This notebook does not alter the two completed approaches.

**Success criterion to investigate:** CER and ICU-segmented WER each at or below 20%. A third approach is useful for the rubric even if its WER does not reach that target. Report the measured result honestly.


In [ ]:
!nvidia-smi
!pip install -q torch torchaudio transformers datasets torchcodec evaluate jiwer pyicu-wheels==2.15.2 accelerate tensorboard soundfile librosa matplotlib av
from pathlib import Path
import json, shutil, subprocess, torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before training.')
%cd /content
if not Path('/content/khmer_asr/.git').exists():
    !git clone https://github.com/Seypa-47/khmer_asr.git
%cd /content/khmer_asr
!git pull --ff-only
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/khmer_asr_final_runs')
ROOT.mkdir(parents=True, exist_ok=True)
THIRD_DIR = ROOT / 'mms-khmer-ctc-frozen'
EVIDENCE_DIR = ROOT / 'results'
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
manifest = json.loads(Path('results/matched_fleurs_split.json').read_text(encoding='utf-8'))
assert tuple(len(manifest['indices'][split]) for split in ('train','validation','test')) == (531,124,114)
print('Shared split loaded; frozen model will be saved at', THIRD_DIR)


## Train and select with validation WER
Only the CTC output head is trainable. The base encoder and adapter remain frozen. Use the same saved FLEURS rows, seed, augmentation, and preprocessing as the tuned MMS comparison. Five epochs are a first controlled run; inspect both validation curves before deciding whether any further run is justified. If Colab disconnects, resume from a complete `checkpoint-*` directory in `THIRD_DIR` with `--resume-from-checkpoint` rather than starting over.


In [ ]:
command = [
    'python', 'src/train_mms.py',
    '--output-dir', str(THIRD_DIR),
    '--model-id', 'facebook/mms-1b-all', '--target-lang', 'khm',
    '--split-manifest', 'results/matched_fleurs_split.json',
    '--metrics-output', 'results/mms_frozen_metrics.json',
    '--trainer-state-output', 'results/mms_frozen_trainer_state.json',
    '--seed', '42', '--num-train-epochs', '5', '--learning-rate', '3e-5',
    '--weight-decay', '0', '--unfreeze-top-layers', '0',
    '--apply-spec-augment', '--lr-scheduler-type', 'cosine',
    '--warmup-steps', '50', '--per-device-train-batch-size', '1',
    '--per-device-eval-batch-size', '1', '--gradient-accumulation-steps', '8',
    '--eval-steps', '50', '--save-steps', '50', '--logging-steps', '10',
    '--selection-metric', 'wer_icu', '--skip-test', '--fp16',
]
subprocess.run(command, check=True)
for name in ('mms_frozen_metrics.json', 'mms_frozen_trainer_state.json'):
    shutil.copy2(Path('results') / name, EVIDENCE_DIR / name)
print('Training completed. Best validation checkpoint is loaded into', THIRD_DIR)


## Evaluate once on the shared test split
Run this after training completes. It compares all three approaches using the same saved test clips and the same CER/WER policy. The first two prediction files come from the existing completed experiments in GitHub. Results are copied to Drive so they survive a Colab disconnect.


In [ ]:
subprocess.run([
    'python', 'src/evaluate_saved_mms.py',
    '--model-dir', str(THIRD_DIR),
    '--split-manifest', 'results/matched_fleurs_split.json',
    '--output', 'results/mms_frozen_predictions.json', '--device', 'cuda',
], check=True)
subprocess.run([
    'python', 'src/score_matched_cer_wer.py',
    '--whisper', 'results/whisper_matched_predictions.json',
    '--mms', 'results/mms_matched_best300_predictions.json',
    '--frozen-mms', 'results/mms_frozen_predictions.json',
    '--output', 'results/matched_cer_wer_three_approaches.json',
], check=True)
subprocess.run([
    'python', 'src/plot_cer_wer.py',
    '--summary', 'results/matched_cer_wer_three_approaches.json',
    '--output', 'results/matched_cer_wer_three_approaches.png',
], check=True)
for name in ('mms_frozen_predictions.json', 'matched_cer_wer_three_approaches.json', 'matched_cer_wer_three_approaches.png'):
    shutil.copy2(Path('results') / name, EVIDENCE_DIR / name)
summary = json.loads(Path('results/matched_cer_wer_three_approaches.json').read_text(encoding='utf-8'))
for name, row in summary['models'].items():
    print(f"{name}: CER {row['cer_percent']:.2f}% | WER {row['wer_percent_icu_segmented']:.2f}%")
